# Recommendation Pipeline V1 (Leakage-safe, Time-based)

## 1) Install Dependencies

In [18]:
# %pip install -q polars implicit faiss-cpu xgboost scikit-learn joblib scipy pandas numpy

## 2) Imports and Config

In [33]:
from pathlib import Path
from datetime import date, timedelta
import gc
import os
import warnings
import numpy as np
import pandas as pd
import polars as pl

from scipy.sparse import csr_matrix
import implicit
import faiss
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
import joblib

warnings.filterwarnings("ignore")

DATA_DIR = Path("/home/chiennc/Big_Data/product")
CHECKPOINT_DIR = Path("/home/chiennc/Big_Data/checkpoint/xgboost")
OUTPUT_DIR = DATA_DIR / "output"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "2019-Oct.csv"
PARQUET_PATH = DATA_DIR / "2019-Oct_zstd.parquet"

EVENT_WEIGHT = {
    "purchase": 5.0,
    "cart": 2.0,
    "view": 0.5,
}

ALS_FACTORS = 64
ALS_ITERATIONS = 20
ALS_REGULARIZATION = 0.1
ALS_USE_GPU = False

CANDIDATE_K = 100
ALS_CANDIDATE_K = 100
ANN_CANDIDATE_K = 200
FINAL_TOPN = 20
MAX_PER_CATEGORY = 2

MAX_USERS_PER_SPLIT = None
SERVE_MAX_USERS = None
SERVE_ACTIVE_USER_START = date(2019, 10, 18)

XGB_MODEL_PATH = CHECKPOINT_DIR / "xgb_model_v3.json"
ENCODER_PATH = CHECKPOINT_DIR / "ordinal_encoder_v3.joblib"
RECS_OUTPUT_PATH = OUTPUT_DIR / "recommendations_top20.parquet"

TRAIN_HISTORY_START = date(2019, 10, 1)
TRAIN_HISTORY_END = date(2019, 10, 17)
TRAIN_LABEL_START = date(2019, 10, 18)
TRAIN_LABEL_END = date(2019, 10, 24)

VAL_HISTORY_END = date(2019, 10, 24)
VAL_LABEL_START = date(2019, 10, 25)
VAL_LABEL_END = date(2019, 10, 28)

TEST_HISTORY_END = date(2019, 10, 28)
TEST_LABEL_START = date(2019, 10, 29)
TEST_LABEL_END = date(2019, 10, 31)

SERVE_CUTOFF = date(2019, 10, 24)

MODEL_FEATURES = [
    "brand",
    "category_code_level1",
    "category_code_level2",
    "event_weekday",
    "user_total_sessions",
    "user_active_days",
    "product_cart_to_purchase_rate",
    "product_total_purchases",
    "product_unique_viewers",
    "activity_count",
    "session_view_count",
    "session_unique_products",
    "price",
    "price_vs_product_avg",
    "user_product_view_count",
]

CATEGORICAL_FEATURES = [
    "brand",
    "event_weekday",
    "category_code_level1",
    "category_code_level2",
]

NUMERIC_FEATURES = [c for c in MODEL_FEATURES if c not in CATEGORICAL_FEATURES]

## 3) Load Data Layer

In [34]:
if not PARQUET_PATH.exists():
    (
        pl.scan_csv(
            CSV_PATH,
            schema_overrides={
                "product_id": pl.Int64,
                "category_id": pl.Int64,
                "user_id": pl.Int64,
                "price": pl.Float32,
                "event_time": pl.Utf8,
            },
        )
        .sink_parquet(PARQUET_PATH, compression="zstd", compression_level=3)
    )

raw_events = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(
        pl.col("user_id").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("user_session").is_not_null()
    )
    .with_columns([
        pl.col("brand").fill_null("unknown").alias("brand"),
        pl.col("category_code").fill_null("unknown").alias("category_code"),
        pl.col("event_time")
        .str.replace(" UTC", "")
        .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
        .alias("event_dt"),
    ])
    .filter(pl.col("event_dt").is_not_null())
    .with_columns(pl.col("event_dt").dt.date().alias("event_date"))
    .sort(["user_id", "event_dt"])
    .collect(streaming=True)
)

print(raw_events.shape)
# print(raw_events.select(pl.col("event_type").value_counts().sort("count", descending=True)))
print(raw_events.select(pl.col("event_type").value_counts()).unnest("event_type").sort("count", descending=True))
print(raw_events.select(pl.min("event_date").alias("min_date"), pl.max("event_date").alias("max_date")))

(42448762, 11)
shape: (3, 2)
┌────────────┬──────────┐
│ event_type ┆ count    │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ view       ┆ 40779399 │
│ cart       ┆ 926514   │
│ purchase   ┆ 742849   │
└────────────┴──────────┘
shape: (1, 2)
┌────────────┬────────────┐
│ min_date   ┆ max_date   │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2019-10-01 ┆ 2019-10-31 │
└────────────┴────────────┘


## 4) Time Windows and Integrity Checks

In [35]:
def assert_date_windows():
    assert TRAIN_HISTORY_START <= TRAIN_HISTORY_END < TRAIN_LABEL_START <= TRAIN_LABEL_END
    assert TRAIN_LABEL_END < VAL_LABEL_START <= VAL_LABEL_END
    assert VAL_LABEL_END < TEST_LABEL_START <= TEST_LABEL_END
    assert TRAIN_HISTORY_START == date(2019, 10, 1)
    assert TEST_LABEL_END == date(2019, 10, 31)

assert_date_windows()

all_days = pl.date_range(
    start=pl.lit(TRAIN_HISTORY_START),
    end=pl.lit(TEST_LABEL_END),
    interval="1d",
    eager=True,
)

print("train_history:", TRAIN_HISTORY_START, TRAIN_HISTORY_END)
print("train_label:", TRAIN_LABEL_START, TRAIN_LABEL_END)
print("val_label:", VAL_LABEL_START, VAL_LABEL_END)
print("test_label:", TEST_LABEL_START, TEST_LABEL_END)
print("total_days_in_scope:", len(all_days))

train_history: 2019-10-01 2019-10-17
train_label: 2019-10-18 2019-10-24
val_label: 2019-10-25 2019-10-28
test_label: 2019-10-29 2019-10-31
total_days_in_scope: 31


## 5) Core Functions (State, ALS, FAISS, Features)

In [36]:
def filter_window(df: pl.DataFrame, start_date: date, end_date: date) -> pl.DataFrame:
    return df.filter(
        (pl.col("event_date") >= pl.lit(start_date))
        & (pl.col("event_date") <= pl.lit(end_date))
    )


def build_interaction_matrix(history_df: pl.DataFrame):
    weighted = (
        history_df
        .filter(pl.col("event_type").is_in(list(EVENT_WEIGHT.keys())))
        .with_columns(
            pl.when(pl.col("event_type") == "purchase")
            .then(pl.lit(EVENT_WEIGHT["purchase"]))
            .when(pl.col("event_type") == "cart")
            .then(pl.lit(EVENT_WEIGHT["cart"]))
            .when(pl.col("event_type") == "view")
            .then(pl.lit(EVENT_WEIGHT["view"]))
            .otherwise(pl.lit(0.0))
            .cast(pl.Float32)
            .alias("weight")
        )
        .group_by(["user_id", "product_id"])
        .agg(pl.col("weight").sum().alias("weight"))
    )

    user_ids = (
        weighted
        .select("user_id")
        .unique()
        .sort("user_id")
        .get_column("user_id")
        .to_list()
    )

    item_ids = (
        weighted
        .select("product_id")
        .unique()
        .sort("product_id")
        .get_column("product_id")
        .to_list()
    )

    user_map_df = pl.DataFrame({"user_id": user_ids, "user_idx": np.arange(len(user_ids), dtype=np.int32)})
    item_map_df = pl.DataFrame({"product_id": item_ids, "item_idx": np.arange(len(item_ids), dtype=np.int32)})

    matrix_df = (
        weighted
        .join(user_map_df, on="user_id", how="inner")
        .join(item_map_df, on="product_id", how="inner")
        .select(["user_idx", "item_idx", "weight"])
    )

    rows = matrix_df.get_column("user_idx").to_numpy()
    cols = matrix_df.get_column("item_idx").to_numpy()
    vals = matrix_df.get_column("weight").to_numpy().astype(np.float32)

    mat = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(item_ids)), dtype=np.float32)

    user_to_idx = dict(zip(user_ids, range(len(user_ids))))
    item_to_idx = dict(zip(item_ids, range(len(item_ids))))
    idx_to_item = np.array(item_ids, dtype=np.int64)

    return mat, user_map_df, item_map_df, user_to_idx, item_to_idx, idx_to_item


def train_als(interaction_csr: csr_matrix):
    als = implicit.als.AlternatingLeastSquares(
        factors=ALS_FACTORS,
        regularization=ALS_REGULARIZATION,
        iterations=ALS_ITERATIONS,
        use_gpu=ALS_USE_GPU,
        random_state=42,
    )
    als.fit(interaction_csr)
    return als


def build_faiss_index(item_factors: np.ndarray):
    vectors = item_factors.astype(np.float32).copy()
    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def build_feature_tables(history_df: pl.DataFrame):
    base = history_df.sort("event_dt")

    user_features = (
        base
        .group_by("user_id")
        .agg([
            pl.col("user_session").n_unique().alias("user_total_sessions"),
            pl.col("event_date").n_unique().alias("user_active_days"),
        ])
    )

    product_features = (
        base
        .group_by("product_id")
        .agg([
            (pl.col("event_type") == "purchase").sum().alias("product_total_purchases"),
            (pl.col("event_type") == "cart").sum().alias("product_total_carts"),
            pl.col("user_id").filter(pl.col("event_type") == "view").n_unique().alias("product_unique_viewers"),
            pl.col("price").mean().alias("product_avg_price"),
        ])
        .with_columns(
            (
                pl.col("product_total_purchases")
                / pl.col("product_total_carts").clip(lower_bound=1)
            ).alias("product_cart_to_purchase_rate")
        )
    )

    session_features = (
        base
        .group_by(["user_id", "user_session"])
        .agg([
            pl.max("event_dt").alias("session_last_dt"),
            pl.len().alias("activity_count"),
            (pl.col("event_type") == "view").sum().alias("session_view_count"),
            pl.col("product_id").n_unique().alias("session_unique_products"),
        ])
    )

    user_recent_session_features = (
        session_features
        .sort(["user_id", "session_last_dt"])
        .group_by("user_id")
        .agg([
            pl.col("activity_count").last().alias("activity_count"),
            pl.col("session_view_count").last().alias("session_view_count"),
            pl.col("session_unique_products").last().alias("session_unique_products"),
        ])
    )

    user_product_features = (
        base
        .filter(pl.col("event_type").is_in(["view", "purchase"]))
        .group_by(["user_id", "product_id"])
        .agg([
            (pl.col("event_type") == "view").sum().alias("user_product_view_count"),
            ((pl.col("event_type") == "purchase").sum() > 0)
            .cast(pl.Int8)
            .alias("user_ever_purchased_product"),
        ])
    )

    product_meta = (
        base
        .group_by("product_id")
        .agg([
            pl.col("brand").last().alias("brand"),
            pl.col("category_code").last().alias("category_code"),
            pl.col("price").last().alias("price"),
        ])
        .with_columns([
            pl.col("category_code").str.split(".").list.get(0).fill_null("unknown").alias("category_code_level1"),
            pl.col("category_code").str.split(".").list.get(1, null_on_oob=True).fill_null("unknown").alias("category_code_level2"),
        ])
    )

    popularity = (
        base
        .filter(pl.col("event_type").is_in(list(EVENT_WEIGHT.keys())))
        .with_columns(pl.when(pl.col("event_type") == "purchase")
            .then(pl.lit(EVENT_WEIGHT["purchase"]))
            .when(pl.col("event_type") == "cart")
            .then(pl.lit(EVENT_WEIGHT["cart"]))
            .when(pl.col("event_type") == "view")
            .then(pl.lit(EVENT_WEIGHT["view"]))
            .otherwise(pl.lit(0.0))
            .cast(pl.Float32)
            .alias("weight"))
        .group_by("product_id")
        .agg(pl.col("weight").sum().alias("pop_score"))
        .sort("pop_score", descending=True)
    )

    user_seen_map = {
        row["user_id"]: set(row["seen_items"])
        for row in (
            base
            .group_by("user_id")
            .agg(pl.col("product_id").unique().alias("seen_items"))
            .iter_rows(named=True)
        )
    }

    user_bought_map = {
        row["user_id"]: set(row["bought_items"])
        for row in (
            base
            .filter(pl.col("event_type") == "purchase")
            .group_by("user_id")
            .agg(pl.col("product_id").unique().alias("bought_items"))
            .iter_rows(named=True)
        )
    }

    return {
        "user_features": user_features,
        "product_features": product_features,
        "user_recent_session_features": user_recent_session_features,
        "user_product_features": user_product_features,
        "product_meta": product_meta,
        "popular_items": popularity.get_column("product_id").to_list(),
        "user_seen_map": user_seen_map,
        "user_bought_map": user_bought_map,
    }


def build_state_asof(events_df: pl.DataFrame, cutoff_date: date):
    history_df = events_df.filter(pl.col("event_date") <= pl.lit(cutoff_date))

    history_max_date = history_df.select(pl.max("event_date")).item()
    assert history_max_date <= cutoff_date

    interaction_csr, user_map_df, item_map_df, user_to_idx, item_to_idx, idx_to_item = build_interaction_matrix(history_df)
    als_model = train_als(interaction_csr)

    user_factors = als_model.user_factors.astype(np.float32)
    item_factors = als_model.item_factors.astype(np.float32)
    ann_index = build_faiss_index(item_factors)

    feature_tables = build_feature_tables(history_df)

    state = {
        "cutoff_date": cutoff_date,
        "history_df": history_df,
        "interaction_csr": interaction_csr,
        "user_map_df": user_map_df,
        "item_map_df": item_map_df,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "idx_to_item": idx_to_item,
        "als_model": als_model,
        "user_factors": user_factors,
        "item_factors": item_factors,
        "ann_index": ann_index,
    }
    state.update(feature_tables)
    return state

## 6) Candidate Retrieval and Ranking Dataset Builder

In [37]:
# def retrieve_candidates_for_users(
#     user_ids,
#     state,
#     candidate_k=CANDIDATE_K,
#     als_k=ALS_CANDIDATE_K,
#     ann_k=ANN_CANDIDATE_K,
# ):
#     records = []

#     user_to_idx = state["user_to_idx"]
#     idx_to_item = state["idx_to_item"]
#     interaction_csr = state["interaction_csr"]

#     for user_id in user_ids:
#         seen_items = state["user_seen_map"].get(user_id, set())

#         blend_score = {}
#         als_raw_score = {}
#         ann_raw_score = {}
#         source_map = {}

#         if user_id in user_to_idx:
#             user_idx = user_to_idx[user_id]

#             als_items, als_scores = state["als_model"].recommend(
#                 user_idx,
#                 interaction_csr[user_idx],
#                 N=als_k,
#                 filter_already_liked_items=True,
#                 recalculate_user=False,
#             )

#             for rank, (item_idx, score) in enumerate(zip(als_items.tolist(), als_scores.tolist()), start=1):
#                 product_id = int(idx_to_item[item_idx])
#                 if product_id in seen_items:
#                     continue
#                 blend_score[product_id] = blend_score.get(product_id, 0.0) + (1.0 / (rank + 1.0))
#                 als_raw_score[product_id] = float(score)
#                 source_map.setdefault(product_id, set()).add("als")

#             query = state["user_factors"][user_idx:user_idx + 1].astype(np.float32).copy()
#             faiss.normalize_L2(query)
#             sim, idx = state["ann_index"].search(query, ann_k)

#             for rank, (item_idx, score) in enumerate(zip(idx[0].tolist(), sim[0].tolist()), start=1):
#                 if item_idx < 0:
#                     continue
#                 product_id = int(idx_to_item[item_idx])
#                 if product_id in seen_items:
#                     continue
#                 blend_score[product_id] = blend_score.get(product_id, 0.0) + (1.0 / (rank + 1.0))
#                 ann_raw_score[product_id] = float(score)
#                 source_map.setdefault(product_id, set()).add("ann")

#         if len(blend_score) < candidate_k:
#             for rank, product_id in enumerate(state["popular_items"], start=1):
#                 if product_id in seen_items:
#                     continue
#                 if product_id in blend_score:
#                     continue
#                 blend_score[product_id] = 1.0 / (1000.0 + rank)
#                 source_map.setdefault(product_id, set()).add("popular")
#                 if len(blend_score) >= candidate_k:
#                     break

#         top_items = sorted(blend_score.items(), key=lambda x: x[1], reverse=True)[:candidate_k]

#         for product_id, retrieval_score in top_items:
#             records.append({
#                 "user_id": int(user_id),
#                 "product_id": int(product_id),
#                 "retrieval_score": float(retrieval_score),
#                 "als_score": float(als_raw_score.get(product_id, 0.0)),
#                 "ann_score": float(ann_raw_score.get(product_id, 0.0)),
#                 "source": "|".join(sorted(source_map.get(product_id, {"popular"}))),
#             })

#     if not records:
#         return pl.DataFrame(schema={
#             "user_id": pl.Int64,
#             "product_id": pl.Int64,
#             "retrieval_score": pl.Float64,
#             "als_score": pl.Float64,
#             "ann_score": pl.Float64,
#             "source": pl.Utf8,
#         })

#     return pl.DataFrame(records)
def retrieve_candidates_for_users(
    user_ids,
    state,
    candidate_k=CANDIDATE_K,
    als_k=ALS_CANDIDATE_K,
    ann_k=ANN_CANDIDATE_K,
):
    records = []
    user_to_idx = state["user_to_idx"]
    idx_to_item = state["idx_to_item"]
    interaction_csr = state["interaction_csr"]
    seen_map = state["user_seen_map"]
    popular_items = state["popular_items"]

    # 1. Lọc các user hợp lệ và lấy index
    valid_users = [u for u in user_ids if u in user_to_idx]
    if not valid_users:
        return pl.DataFrame(schema={
            "user_id": pl.Int64, "product_id": pl.Int64, 
            "retrieval_score": pl.Float64, "als_score": pl.Float64, 
            "ann_score": pl.Float64, "source": pl.Utf8,
        })

    user_indices = np.array([user_to_idx[u] for u in valid_users], dtype=np.int32)

    # 2. Xử lý BATCH: Gọi ALS cho toàn bộ valid users cùng 1 lúc
    als_items_batch, als_scores_batch = state["als_model"].recommend(
        user_indices,
        interaction_csr[user_indices],
        N=als_k,
        filter_already_liked_items=True,
        recalculate_user=False,
    )

    # 3. Xử lý BATCH: Gọi FAISS cho toàn bộ valid users cùng 1 lúc
    queries = state["user_factors"][user_indices].astype(np.float32).copy()
    faiss.normalize_L2(queries)
    ann_scores_batch, ann_items_batch = state["ann_index"].search(queries, ann_k)

    # 4. Gộp kết quả
    for i, user_id in enumerate(valid_users):
        seen_items = seen_map.get(user_id, set())
        blend_score = {}
        als_raw_score = {}
        ann_raw_score = {}
        source_map = {}

        # Ghi nhận ALS
        for rank, (item_idx, score) in enumerate(zip(als_items_batch[i], als_scores_batch[i]), start=1):
            if item_idx < 0: continue # Bỏ qua index padding của implicit
            product_id = int(idx_to_item[item_idx])
            if product_id in seen_items: continue
            blend_score[product_id] = blend_score.get(product_id, 0.0) + (1.0 / (rank + 1.0))
            als_raw_score[product_id] = float(score)
            source_map.setdefault(product_id, set()).add("als")

        # Ghi nhận ANN (FAISS)
        for rank, (item_idx, score) in enumerate(zip(ann_items_batch[i], ann_scores_batch[i]), start=1):
            if item_idx < 0: continue
            product_id = int(idx_to_item[item_idx])
            if product_id in seen_items: continue
            blend_score[product_id] = blend_score.get(product_id, 0.0) + (1.0 / (rank + 1.0))
            ann_raw_score[product_id] = float(score)
            source_map.setdefault(product_id, set()).add("ann")

        # Fallback bằng Popular Items
        if len(blend_score) < candidate_k:
            for rank, product_id in enumerate(popular_items, start=1):
                if product_id in seen_items or product_id in blend_score: continue
                blend_score[product_id] = 1.0 / (1000.0 + rank)
                source_map.setdefault(product_id, set()).add("popular")
                if len(blend_score) >= candidate_k: break

        # Sắp xếp và lưu top K
        top_items = sorted(blend_score.items(), key=lambda x: x[1], reverse=True)[:candidate_k]
        for product_id, retrieval_score in top_items:
            records.append({
                "user_id": int(user_id),
                "product_id": int(product_id),
                "retrieval_score": float(retrieval_score),
                "als_score": float(als_raw_score.get(product_id, 0.0)),
                "ann_score": float(ann_raw_score.get(product_id, 0.0)),
                "source": "|".join(sorted(source_map.get(product_id, {"popular"}))),
            })

    return pl.DataFrame(records)


def _sample_user_ids(user_ids, max_users):
    if max_users is None or len(user_ids) <= max_users:
        return user_ids
    rng = np.random.default_rng(42)
    sampled = rng.choice(np.array(user_ids), size=max_users, replace=False)
    return sampled.tolist()


def build_ranking_dataset(
    events_df: pl.DataFrame,
    state: dict,
    split_name: str,
    label_start: date,
    label_end: date,
    max_users=None,
):
    target_df = filter_window(events_df, label_start, label_end)

    user_ids = target_df.select("user_id").unique().get_column("user_id").to_list()
    user_ids = _sample_user_ids(user_ids, max_users)

    target_df = target_df.filter(pl.col("user_id").is_in(user_ids))

    candidate_df = retrieve_candidates_for_users(user_ids=user_ids, state=state)

    purchase_df = (
        target_df
        .filter(pl.col("event_type") == "purchase")
        .select(["user_id", "product_id"])
        .unique()
        .with_columns(pl.lit(1).cast(pl.Int8).alias("label"))
    )

    candidate_labeled = (
        candidate_df
        .join(purchase_df, on=["user_id", "product_id"], how="left")
        .with_columns(pl.col("label").fill_null(0).cast(pl.Int8))
    )

    missing_positives = (
        purchase_df
        .join(candidate_df.select(["user_id", "product_id"]), on=["user_id", "product_id"], how="anti")
        .with_columns([
            pl.lit(0.0).alias("retrieval_score"),
            pl.lit(0.0).alias("als_score"),
            pl.lit(0.0).alias("ann_score"),
            pl.lit("target_positive").alias("source"),
        ])
    )

    all_pairs = pl.concat([candidate_labeled, missing_positives], how="diagonal_relaxed")

    user_context = (
        target_df
        .group_by("user_id")
        .agg(pl.min("event_dt").alias("context_ts"))
        .with_columns(
            pl.col("context_ts").dt.weekday().cast(pl.Int64).alias("event_weekday")
        )
    )

    rank_df = (
        all_pairs
        .join(user_context, on="user_id", how="left")
        .join(state["product_meta"], on="product_id", how="left")
        .join(state["user_features"], on="user_id", how="left")
        .join(state["product_features"], on="product_id", how="left")
        .join(state["user_recent_session_features"], on="user_id", how="left")
        .join(state["user_product_features"], on=["user_id", "product_id"], how="left")
        .with_columns([
            (pl.col("price") / pl.col("product_avg_price").fill_null(1.0)).alias("price_vs_product_avg"),
            pl.lit(split_name).alias("split"),
            pl.lit(state["cutoff_date"]).alias("snapshot_cutoff"),
            pl.lit(label_start).alias("window_start"),
            pl.lit(label_end).alias("window_end"),
        ])
    )

    for cat_col in ["brand", "category_code_level1", "category_code_level2"]:
        rank_df = rank_df.with_columns(pl.col(cat_col).fill_null("unknown").alias(cat_col))

    for num_col in [
        "user_total_sessions",
        "user_active_days",
        "product_cart_to_purchase_rate",
        "product_total_purchases",
        "product_unique_viewers",
        "activity_count",
        "session_view_count",
        "session_unique_products",
        "price",
        "price_vs_product_avg",
        "user_product_view_count",
    ]:
        rank_df = rank_df.with_columns(pl.col(num_col).fill_null(0).alias(num_col))

    rank_df = rank_df.with_columns([
        pl.col("event_weekday").fill_null(label_start.weekday()).cast(pl.Int64).alias("event_weekday"),
        pl.col("label").fill_null(0).cast(pl.Int8).alias("label"),
    ])

    return rank_df

## 7) Build States and Split Datasets

In [38]:
state_cache = {}

def get_state(cutoff_date: date):
    if cutoff_date not in state_cache:
        print(f"building_state_asof={cutoff_date}")
        state_cache[cutoff_date] = build_state_asof(raw_events, cutoff_date)
    return state_cache[cutoff_date]

# train_state = get_state(TRAIN_HISTORY_END)
# val_state = get_state(VAL_HISTORY_END)
# test_state = get_state(TEST_HISTORY_END)

# train_rank_df = build_ranking_dataset(
#     events_df=raw_events,
#     state=train_state,
#     split_name="train",
#     label_start=TRAIN_LABEL_START,
#     label_end=TRAIN_LABEL_END,
#     max_users=MAX_USERS_PER_SPLIT,
# )

# val_rank_df = build_ranking_dataset(
#     events_df=raw_events,
#     state=val_state,
#     split_name="val",
#     label_start=VAL_LABEL_START,
#     label_end=VAL_LABEL_END,
#     max_users=MAX_USERS_PER_SPLIT,
# )

# test_rank_df = build_ranking_dataset(
#     events_df=raw_events,
#     state=test_state,
#     split_name="test",
#     label_start=TEST_LABEL_START,
#     label_end=TEST_LABEL_END,
#     max_users=MAX_USERS_PER_SPLIT,
# )

# print("train_rank_df", train_rank_df.shape, "positive", train_rank_df.filter(pl.col("label") == 1).height)
# print("val_rank_df", val_rank_df.shape, "positive", val_rank_df.filter(pl.col("label") == 1).height)
# print("test_rank_df", test_rank_df.shape, "positive", test_rank_df.filter(pl.col("label") == 1).height)

## 8) Temporal and Leakage Checks

In [39]:
# def assert_no_leakage(rank_df: pl.DataFrame, split_name: str):
#     ok = rank_df.select((pl.col("snapshot_cutoff") < pl.col("window_start")).all()).item()
#     assert ok, f"leakage_detected_in_{split_name}"

# assert_no_leakage(train_rank_df, "train")
# assert_no_leakage(val_rank_df, "val")
# assert_no_leakage(test_rank_df, "test")

# assert TRAIN_LABEL_END < VAL_LABEL_START
# assert VAL_LABEL_END < TEST_LABEL_START

# assert train_state["history_df"].select(pl.max("event_date")).item() <= TRAIN_HISTORY_END
# assert val_state["history_df"].select(pl.max("event_date")).item() <= VAL_HISTORY_END
# assert test_state["history_df"].select(pl.max("event_date")).item() <= TEST_HISTORY_END

# print("temporal_checks=passed")

## 9) Prepare Matrices for XGBoost

In [40]:
# def to_model_pandas(rank_df: pl.DataFrame) -> pd.DataFrame:
#     cols = ["user_id", "product_id", "label", "split"] + MODEL_FEATURES + [
#         "retrieval_score", "als_score", "ann_score", "source",
#         "snapshot_cutoff", "window_start", "window_end"
#     ]
#     pdf = rank_df.select(cols).to_pandas()

#     for c in CATEGORICAL_FEATURES:
#         pdf[c] = pdf[c].astype(str).fillna("unknown")

#     for c in NUMERIC_FEATURES + ["retrieval_score", "als_score", "ann_score"]:
#         pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)

#     pdf["label"] = pdf["label"].astype(int)
#     return pdf

# train_pdf = to_model_pandas(train_rank_df)
# val_pdf = to_model_pandas(val_rank_df)
# test_pdf = to_model_pandas(test_rank_df)

# encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# X_train = train_pdf[MODEL_FEATURES].copy()
# X_val = val_pdf[MODEL_FEATURES].copy()
# X_test = test_pdf[MODEL_FEATURES].copy()

# y_train = train_pdf["label"].to_numpy()
# y_val = val_pdf["label"].to_numpy()
# y_test = test_pdf["label"].to_numpy()

# X_train[CATEGORICAL_FEATURES] = encoder.fit_transform(X_train[CATEGORICAL_FEATURES])
# X_val[CATEGORICAL_FEATURES] = encoder.transform(X_val[CATEGORICAL_FEATURES])
# X_test[CATEGORICAL_FEATURES] = encoder.transform(X_test[CATEGORICAL_FEATURES])

# joblib.dump(encoder, ENCODER_PATH)
# print(f"saved_encoder={ENCODER_PATH}")
# print(X_train.shape, X_val.shape, X_test.shape)

## 10) Train and Evaluate XGBoost

In [41]:
# neg = (y_train == 0).sum()
# pos = (y_train == 1).sum()
# scale_pos_weight = float(neg / max(pos, 1))

# xgb_model = XGBClassifier(
#     n_estimators=1400,
#     learning_rate=0.05,
#     max_depth=6,
#     min_child_weight=2,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_alpha=0.0,
#     reg_lambda=1.0,
#     tree_method="hist",
#     objective="binary:logistic",
#     eval_metric="logloss",
#     early_stopping_rounds=80,
#     random_state=42,
#     n_jobs=-1,
#     scale_pos_weight=scale_pos_weight,
# )

# xgb_model.fit(
#     X_train,
#     y_train,
#     eval_set=[(X_train, y_train), (X_val, y_val)],
#     verbose=100,
# )

# xgb_model.save_model(XGB_MODEL_PATH)
# print(f"saved_model={XGB_MODEL_PATH}")


# def eval_binary(y_true, y_prob, split_name):
#     y_prob = np.clip(y_prob, 1e-7, 1 - 1e-7)
#     metrics = {
#         "split": split_name,
#         "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
#         "pr_auc": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
#         "logloss": float(log_loss(y_true, y_prob)),
#         "positives": int((y_true == 1).sum()),
#         "samples": int(len(y_true)),
#     }
#     return metrics

# val_prob = xgb_model.predict_proba(X_val)[:, 1]
# test_prob = xgb_model.predict_proba(X_test)[:, 1]

# metrics_val = eval_binary(y_val, val_prob, "val")
# metrics_test = eval_binary(y_test, test_prob, "test")

# print(metrics_val)
# print(metrics_test)

## 11) Serving Pipeline (as-of 24/10) + Re-ranking

In [42]:
# serve_state = get_state(SERVE_CUTOFF)

# serve_users = (
#     serve_state["history_df"]
#     .filter(pl.col("event_date") >= pl.lit(SERVE_ACTIVE_USER_START))
#     .select("user_id")
#     .unique()
#     .get_column("user_id")
#     .to_list()
# )

# serve_users = _sample_user_ids(serve_users, SERVE_MAX_USERS)

# serve_candidates = retrieve_candidates_for_users(
#     user_ids=serve_users,
#     state=serve_state,
#     candidate_k=CANDIDATE_K,
# )

# serve_context = (
#     serve_state["history_df"]
#     .group_by("user_id")
#     .agg(pl.max("event_dt").alias("context_ts"))
#     .with_columns(pl.col("context_ts").dt.weekday().cast(pl.Int64).alias("event_weekday"))
# )

# serve_features = (
#     serve_candidates
#     .join(serve_context, on="user_id", how="left")
#     .join(serve_state["product_meta"], on="product_id", how="left")
#     .join(serve_state["user_features"], on="user_id", how="left")
#     .join(serve_state["product_features"], on="product_id", how="left")
#     .join(serve_state["user_recent_session_features"], on="user_id", how="left")
#     .join(serve_state["user_product_features"], on=["user_id", "product_id"], how="left")
#     .with_columns((pl.col("price") / pl.col("product_avg_price").fill_null(1.0)).alias("price_vs_product_avg"))
# )

# for cat_col in ["brand", "category_code_level1", "category_code_level2"]:
#     serve_features = serve_features.with_columns(pl.col(cat_col).fill_null("unknown").alias(cat_col))

# for num_col in [
#     "user_total_sessions",
#     "user_active_days",
#     "product_cart_to_purchase_rate",
#     "product_total_purchases",
#     "product_unique_viewers",
#     "activity_count",
#     "session_view_count",
#     "session_unique_products",
#     "price",
#     "price_vs_product_avg",
#     "user_product_view_count",
# ]:
#     serve_features = serve_features.with_columns(pl.col(num_col).fill_null(0).alias(num_col))

# serve_features = serve_features.with_columns(
#     pl.col("event_weekday").fill_null(SERVE_CUTOFF.weekday()).cast(pl.Int64).alias("event_weekday")
# )

# serve_pdf = serve_features.select(
#     ["user_id", "product_id", "retrieval_score", "source", "category_code_level1"] + MODEL_FEATURES
# ).to_pandas()

# for c in CATEGORICAL_FEATURES:
#     serve_pdf[c] = serve_pdf[c].astype(str).fillna("unknown")
# for c in NUMERIC_FEATURES + ["retrieval_score"]:
#     serve_pdf[c] = pd.to_numeric(serve_pdf[c], errors="coerce").fillna(0.0)

# serve_X = serve_pdf[MODEL_FEATURES].copy()
# serve_X[CATEGORICAL_FEATURES] = encoder.transform(serve_X[CATEGORICAL_FEATURES])
# serve_pdf["score"] = xgb_model.predict_proba(serve_X)[:, 1]


# def rerank_balanced(pdf: pd.DataFrame, user_bought_map: dict, topn: int = 20, max_per_category: int = 2):
#     rows = []
#     sorted_pdf = pdf.sort_values(["user_id", "score", "retrieval_score"], ascending=[True, False, False])

#     for user_id, group in sorted_pdf.groupby("user_id", sort=False):
#         bought = user_bought_map.get(int(user_id), set())
#         cat_counter = {}
#         rank = 1

#         for _, row in group.iterrows():
#             product_id = int(row["product_id"])
#             if product_id in bought:
#                 continue

#             cat = row["category_code_level1"] if pd.notna(row["category_code_level1"]) else "unknown"
#             cat = str(cat)
#             if cat_counter.get(cat, 0) >= max_per_category:
#                 continue

#             rows.append({
#                 "user_id": int(user_id),
#                 "rank": rank,
#                 "product_id": product_id,
#                 "score": float(row["score"]),
#                 "retrieval_score": float(row["retrieval_score"]),
#                 "source": str(row["source"]),
#             })
#             cat_counter[cat] = cat_counter.get(cat, 0) + 1
#             rank += 1

#             if rank > topn:
#                 break

#     return pd.DataFrame(rows, columns=["user_id", "rank", "product_id", "score", "retrieval_score", "source"])

# recs_pdf = rerank_balanced(
#     pdf=serve_pdf,
#     user_bought_map=serve_state["user_bought_map"],
#     topn=FINAL_TOPN,
#     max_per_category=MAX_PER_CATEGORY,
# )

# assert recs_pdf.groupby("user_id")["rank"].max().fillna(0).max() <= FINAL_TOPN

# pl.from_pandas(recs_pdf).write_parquet(RECS_OUTPUT_PATH)
# print(f"saved_recommendations={RECS_OUTPUT_PATH}")
# print(recs_pdf.head(20))

## 12) Final Output Checks

In [43]:
# result = pl.read_parquet(RECS_OUTPUT_PATH)
# expected_cols = ["user_id", "rank", "product_id", "score", "retrieval_score", "source"]

# assert result.columns == expected_cols
# assert result.select(pl.col("rank").max()).item() <= FINAL_TOPN

# print(result.shape)
# print(result.head(10))

## 13) GPU Batched Pipeline


In [44]:
GPU_BATCH_USERS = 12000

def assert_no_leakage(rank_df: pl.DataFrame, split_name: str):
    ok = rank_df.select((pl.col("snapshot_cutoff") < pl.col("window_start")).all()).item()
    assert ok, f"leakage_detected_in_{split_name}"


def to_model_pandas(rank_df: pl.DataFrame) -> pd.DataFrame:
    cols = ["user_id", "product_id", "label", "split"] + MODEL_FEATURES + [
        "retrieval_score", "als_score", "ann_score", "source",
        "snapshot_cutoff", "window_start", "window_end"
    ]
    pdf = rank_df.select(cols).to_pandas()

    for c in CATEGORICAL_FEATURES:
        pdf[c] = pdf[c].astype(str).fillna("unknown")

    for c in NUMERIC_FEATURES + ["retrieval_score", "als_score", "ann_score"]:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)

    pdf["label"] = pdf["label"].astype(int)
    return pdf


def eval_binary(y_true, y_prob, split_name):
    y_prob = np.clip(y_prob, 1e-7, 1 - 1e-7)
    metrics = {
        "split": split_name,
        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        "logloss": float(log_loss(y_true, y_prob)),
        "positives": int((y_true == 1).sum()),
        "samples": int(len(y_true)),
    }
    return metrics

GPU_SERVE_BATCH_USERS = 12000
GPU_MAX_USERS_PER_SPLIT = None
GPU_SERVE_MAX_USERS = SERVE_MAX_USERS
GPU_NEG_POS_RATIO = 4
GPU_MIN_NEG_PER_BATCH = 20000
GPU_SEED = 42

XGB_GPU_MODEL_PATH = CHECKPOINT_DIR / "xgb_model_v3_gpu.json"
ENCODER_GPU_PATH = CHECKPOINT_DIR / "ordinal_encoder_v3_gpu.joblib"
RECS_GPU_OUTPUT_PATH = OUTPUT_DIR / "recommendations_top20_gpu.parquet"


def iter_user_batches(user_ids, batch_size):
    for i in range(0, len(user_ids), batch_size):
        yield user_ids[i:i + batch_size]


def rebalance_rank_batch(rank_df: pl.DataFrame, neg_pos_ratio=4, min_neg_per_batch=20000):
    pos_df = rank_df.filter(pl.col("label") == 1)
    neg_df = rank_df.filter(pl.col("label") == 0)

    if neg_df.height == 0:
        return rank_df

    target_neg = max(min_neg_per_batch, pos_df.height * neg_pos_ratio)
    target_neg = min(target_neg, neg_df.height)

    if target_neg < neg_df.height:
        neg_df = neg_df.sample(n=target_neg, seed=GPU_SEED, shuffle=True)

    return pl.concat([pos_df, neg_df], how="vertical_relaxed")


# def build_ranking_dataset_batched(
#     events_df: pl.DataFrame,
#     state: dict,
#     split_name: str,
#     label_start: date,
#     label_end: date,
#     batch_users: int,
#     max_users=None,
# ):
#     target_full = filter_window(events_df, label_start, label_end)
#     user_ids = target_full.select("user_id").unique().get_column("user_id").to_list()
#     user_ids = _sample_user_ids(user_ids, max_users)

#     batch_frames = []

#     for users_chunk in iter_user_batches(user_ids, batch_users):
#         chunk_target = target_full.filter(pl.col("user_id").is_in(users_chunk))
#         chunk_rank = build_ranking_dataset(
#             events_df=chunk_target,
#             state=state,
#             split_name=split_name,
#             label_start=label_start,
#             label_end=label_end,
#             max_users=None,
#         )
#         chunk_rank = rebalance_rank_batch(
#             rank_df=chunk_rank,
#             neg_pos_ratio=GPU_NEG_POS_RATIO,
#             min_neg_per_batch=GPU_MIN_NEG_PER_BATCH,
#         )
#         batch_frames.append(chunk_rank)

#     if not batch_frames:
#         return pl.DataFrame()

#     return pl.concat(batch_frames, how="vertical_relaxed")
def build_ranking_dataset_batched(
    events_df: pl.DataFrame,
    state: dict,
    split_name: str,
    label_start: date,
    label_end: date,
    batch_users: int,
    max_users=None,
):
    target_full = filter_window(events_df, label_start, label_end)
    user_ids = target_full.select("user_id").unique().get_column("user_id").to_list()
    user_ids = _sample_user_ids(user_ids, max_users)

    # Tạo thư mục chứa các batch tạm
    temp_dir = OUTPUT_DIR / f"temp_batches_{split_name}"
    temp_dir.mkdir(parents=True, exist_ok=True)
    batch_files = []

    for i, users_chunk in enumerate(iter_user_batches(user_ids, batch_users)):
        chunk_target = target_full.filter(pl.col("user_id").is_in(users_chunk))
        
        chunk_rank = build_ranking_dataset(
            events_df=chunk_target,
            state=state,
            split_name=split_name,
            label_start=label_start,
            label_end=label_end,
            max_users=None,
        )
        
        chunk_rank = rebalance_rank_batch(
            rank_df=chunk_rank,
            neg_pos_ratio=GPU_NEG_POS_RATIO,
            min_neg_per_batch=GPU_MIN_NEG_PER_BATCH,
        )
        
        # Lưu ra ổ cứng thay vì nhét vào List
        file_path = temp_dir / f"batch_{i}.parquet"
        chunk_rank.write_parquet(file_path)
        batch_files.append(str(file_path))
        
        # Giải phóng bộ nhớ RAM triệt để
        del chunk_rank
        del chunk_target
        gc.collect()

    if not batch_files:
        return pl.DataFrame()

    # Quét tất cả các file đã lưu và nối lại (tiết kiệm RAM)
    final_df = pl.scan_parquet(batch_files).collect(streaming=True)
    return final_df


## 14) GPU Batched States and Datasets


In [45]:
train_state_gpu = get_state(TRAIN_HISTORY_END)
val_state_gpu = get_state(VAL_HISTORY_END)
test_state_gpu = get_state(TEST_HISTORY_END)

train_rank_gpu = build_ranking_dataset_batched(
    events_df=raw_events,
    state=train_state_gpu,
    split_name="train",
    label_start=TRAIN_LABEL_START,
    label_end=TRAIN_LABEL_END,
    batch_users=GPU_BATCH_USERS,
    max_users=GPU_MAX_USERS_PER_SPLIT,
)

val_rank_gpu = build_ranking_dataset_batched(
    events_df=raw_events,
    state=val_state_gpu,
    split_name="val",
    label_start=VAL_LABEL_START,
    label_end=VAL_LABEL_END,
    batch_users=GPU_BATCH_USERS,
    max_users=GPU_MAX_USERS_PER_SPLIT,
)

test_rank_gpu = build_ranking_dataset_batched(
    events_df=raw_events,
    state=test_state_gpu,
    split_name="test",
    label_start=TEST_LABEL_START,
    label_end=TEST_LABEL_END,
    batch_users=GPU_BATCH_USERS,
    max_users=GPU_MAX_USERS_PER_SPLIT,
)

assert_no_leakage(train_rank_gpu, "train_gpu")
assert_no_leakage(val_rank_gpu, "val_gpu")
assert_no_leakage(test_rank_gpu, "test_gpu")

train_pdf_gpu = to_model_pandas(train_rank_gpu)
val_pdf_gpu = to_model_pandas(val_rank_gpu)
test_pdf_gpu = to_model_pandas(test_rank_gpu)

encoder_gpu = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_train_gpu = train_pdf_gpu[MODEL_FEATURES].copy()
X_val_gpu = val_pdf_gpu[MODEL_FEATURES].copy()
X_test_gpu = test_pdf_gpu[MODEL_FEATURES].copy()

y_train_gpu = train_pdf_gpu["label"].to_numpy()
y_val_gpu = val_pdf_gpu["label"].to_numpy()
y_test_gpu = test_pdf_gpu["label"].to_numpy()

X_train_gpu[CATEGORICAL_FEATURES] = encoder_gpu.fit_transform(X_train_gpu[CATEGORICAL_FEATURES])
X_val_gpu[CATEGORICAL_FEATURES] = encoder_gpu.transform(X_val_gpu[CATEGORICAL_FEATURES])
X_test_gpu[CATEGORICAL_FEATURES] = encoder_gpu.transform(X_test_gpu[CATEGORICAL_FEATURES])

joblib.dump(encoder_gpu, ENCODER_GPU_PATH)



building_state_asof=2019-10-17


100%|██████████| 20/20 [06:56<00:00, 20.80s/it]


building_state_asof=2019-10-24


100%|██████████| 20/20 [09:03<00:00, 27.16s/it]


building_state_asof=2019-10-28


100%|██████████| 20/20 [09:56<00:00, 29.83s/it]


['/home/chiennc/Big_Data/checkpoint/xgboost/ordinal_encoder_v3_gpu.joblib']

## 15) GPU Train and Evaluate XGBoost


In [46]:
neg_gpu = (y_train_gpu == 0).sum()
pos_gpu = (y_train_gpu == 1).sum()
spw_gpu = float(neg_gpu / max(pos_gpu, 1))

xgb_gpu_common = dict(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    early_stopping_rounds=80,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=spw_gpu,
)

try:
    xgb_gpu = XGBClassifier(
        **xgb_gpu_common,
        tree_method="hist",
        device="cuda",
        predictor="gpu_predictor",
        max_bin=256,
    )
    xgb_gpu.fit(
        X_train_gpu,
        y_train_gpu,
        eval_set=[(X_train_gpu, y_train_gpu), (X_val_gpu, y_val_gpu)],
        verbose=100,
    )
except TypeError:
    xgb_gpu = XGBClassifier(
        **xgb_gpu_common,
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        max_bin=256,
    )
    xgb_gpu.fit(
        X_train_gpu,
        y_train_gpu,
        eval_set=[(X_train_gpu, y_train_gpu), (X_val_gpu, y_val_gpu)],
        verbose=100,
    )

xgb_gpu.save_model(XGB_GPU_MODEL_PATH)

val_prob_gpu = xgb_gpu.predict_proba(X_val_gpu)[:, 1]
test_prob_gpu = xgb_gpu.predict_proba(X_test_gpu)[:, 1]

metrics_val_gpu = eval_binary(y_val_gpu, val_prob_gpu, "val_gpu")
metrics_test_gpu = eval_binary(y_test_gpu, test_prob_gpu, "test_gpu")

print(metrics_val_gpu)
print(metrics_test_gpu)
print(f"saved_model={XGB_GPU_MODEL_PATH}")
print(f"saved_encoder={ENCODER_GPU_PATH}")


[0]	validation_0-logloss:0.66941	validation_1-logloss:0.67041
[100]	validation_0-logloss:0.29993	validation_1-logloss:0.34515
[200]	validation_0-logloss:0.28120	validation_1-logloss:0.33582
[300]	validation_0-logloss:0.27076	validation_1-logloss:0.33219
[400]	validation_0-logloss:0.26379	validation_1-logloss:0.33007
[500]	validation_0-logloss:0.25864	validation_1-logloss:0.32836
[600]	validation_0-logloss:0.25407	validation_1-logloss:0.32731
[700]	validation_0-logloss:0.24997	validation_1-logloss:0.32785
[736]	validation_0-logloss:0.24857	validation_1-logloss:0.32757
{'split': 'val_gpu', 'roc_auc': 0.914768271197498, 'pr_auc': 0.771139770171636, 'logloss': 0.327256530468172, 'positives': 75376, 'samples': 995376}
{'split': 'test_gpu', 'roc_auc': 0.9112978617467684, 'pr_auc': 0.7541684907861022, 'logloss': 0.27555466729795186, 'positives': 51509, 'samples': 751509}
saved_model=/home/chiennc/Big_Data/checkpoint/xgboost/xgb_model_v3_gpu.json
saved_encoder=/home/chiennc/Big_Data/checkpoint

## 16) GPU Serving Functions


In [49]:
def rerank_balanced_gpu(pdf: pd.DataFrame, user_bought_map: dict, topn: int = 20, max_per_category: int = 2):
    rows = []
    sorted_pdf = pdf.sort_values(["user_id", "score", "retrieval_score"], ascending=[True, False, False])

    for user_id, group in sorted_pdf.groupby("user_id", sort=False):
        bought = user_bought_map.get(int(user_id), set())
        cat_counter = {}
        rank = 1

        for _, row in group.iterrows():
            product_id = int(row["product_id"])
            if product_id in bought:
                continue

            cat = row["category_code_level1"] if pd.notna(row["category_code_level1"]) else "unknown"
            cat = str(cat)
            if cat_counter.get(cat, 0) >= max_per_category:
                continue

            rows.append({
                "user_id": int(user_id),
                "rank": rank,
                "product_id": product_id,
                "score": float(row["score"]),
                "retrieval_score": float(row["retrieval_score"]),
                "source": str(row["source"]),
            })
            cat_counter[cat] = cat_counter.get(cat, 0) + 1
            rank += 1

            if rank > topn:
                break

    return pd.DataFrame(rows, columns=["user_id", "rank", "product_id", "score", "retrieval_score", "source"])


def score_serve_batch(state, model, encoder, users_chunk):
    cand = retrieve_candidates_for_users(
        user_ids=users_chunk,
        state=state,
        candidate_k=CANDIDATE_K,
    )

    if cand.height == 0:
        return pd.DataFrame(columns=["user_id", "rank", "product_id", "score", "retrieval_score", "source"])

    ctx = (
        state["history_df"]
        .filter(pl.col("user_id").is_in(users_chunk))
        .group_by("user_id")
        .agg(pl.max("event_dt").alias("context_ts"))
        .with_columns(pl.col("context_ts").dt.weekday().cast(pl.Int64).alias("event_weekday"))
    )

    feat = (
        cand
        .join(ctx, on="user_id", how="left")
        .join(state["product_meta"], on="product_id", how="left")
        .join(state["user_features"], on="user_id", how="left")
        .join(state["product_features"], on="product_id", how="left")
        .join(state["user_recent_session_features"], on="user_id", how="left")
        .join(state["user_product_features"], on=["user_id", "product_id"], how="left")
        .with_columns((pl.col("price") / pl.col("product_avg_price").fill_null(1.0)).alias("price_vs_product_avg"))
    )

    for cat_col in ["brand", "category_code_level1", "category_code_level2"]:
        feat = feat.with_columns(pl.col(cat_col).fill_null("unknown").alias(cat_col))

    for num_col in [
        "user_total_sessions",
        "user_active_days",
        "product_cart_to_purchase_rate",
        "product_total_purchases",
        "product_unique_viewers",
        "activity_count",
        "session_view_count",
        "session_unique_products",
        "price",
        "price_vs_product_avg",
        "user_product_view_count",
    ]:
        feat = feat.with_columns(pl.col(num_col).fill_null(0).alias(num_col))

    feat = feat.with_columns(pl.col("event_weekday").fill_null(SERVE_CUTOFF.weekday()).cast(pl.Int64).alias("event_weekday"))

    serve_pdf_batch = feat.select(
        ["user_id", "product_id", "retrieval_score", "source"] + MODEL_FEATURES
    ).to_pandas()

    for c in CATEGORICAL_FEATURES:
        serve_pdf_batch[c] = serve_pdf_batch[c].astype(str).fillna("unknown")

    for c in NUMERIC_FEATURES + ["retrieval_score"]:
        serve_pdf_batch[c] = pd.to_numeric(serve_pdf_batch[c], errors="coerce").fillna(0.0)

    X_batch = serve_pdf_batch[MODEL_FEATURES].copy()
    X_batch[CATEGORICAL_FEATURES] = encoder.transform(X_batch[CATEGORICAL_FEATURES])
    serve_pdf_batch["score"] = model.predict_proba(X_batch)[:, 1]

    recs_batch = rerank_balanced_gpu(
        pdf=serve_pdf_batch,
        user_bought_map=state["user_bought_map"],
        topn=FINAL_TOPN,
        max_per_category=MAX_PER_CATEGORY,
    )
    return recs_batch


## 17) GPU Serving and Output


In [50]:
serve_state_gpu = get_state(SERVE_CUTOFF)
serve_users_gpu = (
    serve_state_gpu["history_df"]
    .filter(pl.col("event_date") >= pl.lit(SERVE_ACTIVE_USER_START))
    .select("user_id")
    .unique()
    .get_column("user_id")
    .to_list()
)
serve_users_gpu = _sample_user_ids(serve_users_gpu, GPU_SERVE_MAX_USERS)

serve_batches = []
for users_chunk in iter_user_batches(serve_users_gpu, GPU_SERVE_BATCH_USERS):
    serve_batches.append(score_serve_batch(serve_state_gpu, xgb_gpu, encoder_gpu, users_chunk))

recs_gpu_pdf = pd.concat(serve_batches, ignore_index=True) if serve_batches else pd.DataFrame(
    columns=["user_id", "rank", "product_id", "score", "retrieval_score", "source"]
)

assert recs_gpu_pdf.groupby("user_id")["rank"].max().fillna(0).max() <= FINAL_TOPN

pl.from_pandas(recs_gpu_pdf).write_parquet(RECS_GPU_OUTPUT_PATH)
print(f"saved_recommendations={RECS_GPU_OUTPUT_PATH}")
print(recs_gpu_pdf.head(20))


saved_recommendations=/home/chiennc/Big_Data/product/output/recommendations_top20_gpu.parquet
      user_id  rank  product_id     score  retrieval_score source
0    33869381     1     1004836  0.916235         0.142857    als
1    33869381     2     1004857  0.405691         0.031250    als
2    33869381     3    12710856  0.401626         0.041667    ann
3    33869381     4     6902456  0.400121         0.023256    ann
4    33869381     5     6902498  0.376365         0.071429    ann
5    33869381     6    28712327  0.357774         0.058824    als
6    33869381     7    28713027  0.341813         0.026316    als
7    33869381     8    12703415  0.304831         0.038462    als
8    33869381     9     7006333  0.266046         0.058824    ann
9    33869381    10     7004554  0.246281         0.125000    ann
10   33869381    11    12400121  0.227552         0.024390    als
11   33869381    12     6800653  0.208004         0.022222    als
12   33869381    13    19200005  0.207142       